In [1]:
##### Converts raw rasters on cropland and crop production into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # $ of crop production per HA of cropland 

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# cropland raster
cropland = f"{cd}/Data/Raw/Predictors/Ag_land_ME/crop_land_ha_2020.tif"

# crop production raster
crop_production = f"{cd}/Data/Clean/Production/crop_production_USD_2020.tif"

# save paths
pixels = f"{cd}/Data/Clean/Predictors/Rasters/USD_production_per_HA.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/USD_production_per_HA.csv"

In [6]:
#### Run resampling for pixel scale 

### PREP 
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()

### RESAMPLE 
def resample_to_ref(src_path):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            src_nodata=src.nodata,   
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

land       = resample_to_ref(cropland)
production = resample_to_ref(crop_production)

### CALCULATE 
with np.errstate(invalid="ignore", divide="ignore"):
    intensity = np.where(
        (land > 0) & ~np.isnan(land) & ~np.isnan(production),
        production / land,
        np.nan
    )

# clip at 99th percentile
p99 = np.nanpercentile(intensity, 99)
intensity = np.clip(intensity, 0, p99)

### SAVE 
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = intensity.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

p99: 102547890176.00 — capping values above this


In [8]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE

# sum irrigation and cropland in each vector
zonal_production  = rasterstats.zonal_stats(gdf_proj, crop_production, stats="sum", nodata=-9999)
zonal_land = rasterstats.zonal_stats(gdf_proj, cropland,   stats="sum", nodata=-9999)

### compute share at vector level
production_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_production])
land_sums = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_land])

with np.errstate(invalid="ignore", divide="ignore"):
    pct = np.where(
        (land_sums > 0) & ~np.isnan(land_sums) & ~np.isnan(production_sums),
        production_sums / land_sums,
        np.nan
    )

result["USD_production_per_HA"] = pct

### save
result.to_csv(vectors, index=False)